# 00 — Pre-Annotation Otomatis (GroundingDINO + SAM) dari 3 Subfolder (`kaos`, `jaket`, `kemeja`)
**Runtime:** GPU T4 (Colab Free) -> `Runtime > Change runtime type > T4 GPU`.

Notebook ini membantu mempercepat proses anotasi untuk **test set HP Anda** (40+ foto) yang tersimpan di Google Drive dalam 3 subfolder (`kaos`, `jaket`, `kemeja`):
1. Membaca seluruh foto dari 3 subfolder Anda.
2. Mendeteksi lokasi objek pakaian menggunakan **GroundingDINO-tiny** (sesuai jenis pakaian tiap subfolder).
3. Membuat mask poligon rapi menggunakan **Segment Anything Model (`SAM-vit-base`)** — *pure* HuggingFace Transformers (tanpa compile error).
4. Menyederhanakan titik poligon (`approxPolyDP`) agar ringan dan mudah dirapikan di **Roboflow**.
5. Menghasilkan folder `preannotated_for_roboflow/` berisi **semua foto + `_annotations.coco.json`** yang siap di-upload langsung ke Roboflow untuk Anda cek/rapikan & ambil screenshot anotasinya.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers torch torchvision opencv-python-headless Pillow tqdm

import os, glob, json, shutil
import cv2
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, SamModel, SamProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## 1. Konfigurasi Path Folder Google Drive & Nama Kelas
- Ubah `SOURCE_ROOT_DIR` ke lokasi folder di Google Drive Anda yang berisi 3 subfolder (`kaos`, `jaket`, `kemeja`).
- **PENTING:** Pastikan nilai *value* pada `FOLDER_TO_CLASS` **persis sama** dengan nama kelas di dataset latih Anda di Roboflow (misal jika di dataset latih namanya `baju`, ubah `'kemeja': 'baju'`).


In [ ]:
# === GANTI PATH INI SESUAI FOLDER DI GOOGLE DRIVE ANDA ===
SOURCE_ROOT_DIR = '/content/drive/MyDrive/FOLDER_FOTO_HP_ANDA'

# Lokasi output siap upload ke Roboflow
OUTPUT_DIR  = '/content/drive/MyDrive/CPMK3_Instance_Segmentation/preannotated_for_roboflow'
PREVIEW_DIR = '/content/drive/MyDrive/CPMK3_Instance_Segmentation/preannotated_preview'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PREVIEW_DIR, exist_ok=True)

# Pemetaan nama subfolder -> (nama_kelas_di_roboflow, prompt_deteksi_inggris)
# Sesuaikan nama kelas agar 100% sama dengan dataset latih Anda!
FOLDER_CONFIG = {
    'kaos':   {'class_name': 'kaos',   'prompt': 't-shirt. shirt. top.'},
    'jaket':  {'class_name': 'jaket',  'prompt': 'jacket. coat. hoodie. outerwear.'},
    'kemeja': {'class_name': 'kemeja', 'prompt': 'shirt. button-up shirt. collared shirt.'},
}

# Cek isi folder Anda
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
found_files = []
for sub in os.listdir(SOURCE_ROOT_DIR) if os.path.exists(SOURCE_ROOT_DIR) else []:
    sub_path = os.path.join(SOURCE_ROOT_DIR, sub)
    if not os.path.isdir(sub_path):
        continue
    key = sub.strip().lower()
    if key not in FOLDER_CONFIG:
        print(f'[INFO] Subfolder diabaikan (tidak ada di FOLDER_CONFIG): {sub}')
        continue
    imgs = [os.path.join(sub_path, f) for f in sorted(os.listdir(sub_path)) if f.endswith(IMG_EXTS)]
    print(f'Subfolder "{sub}" -> kelas "{FOLDER_CONFIG[key]["class_name"]}": {len(imgs)} foto')
    for p in imgs:
        found_files.append((p, key))

print(f'Total foto yang akan di-pre-annotate: {len(found_files)}')


## 2. Muat Model GroundingDINO-tiny + SAM-vit-base


In [ ]:
DINO_ID = 'IDEA-Research/grounding-dino-tiny'
SAM_ID  = 'facebook/sam-vit-base'

print('Memuat GroundingDINO...')
dino_processor = AutoProcessor.from_pretrained(DINO_ID)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_ID).to(device).eval()

print('Memuat SAM (Segment Anything)...')
sam_processor = SamProcessor.from_pretrained(SAM_ID)
sam_model = SamModel.from_pretrained(SAM_ID).to(device).eval()

print('Kedua model siap!')


## 3. Jalankan Pre-Annotation & Ekspor ke Format COCO Segmentation JSON
- Jika dalam 1 foto hanya ada 1 pakaian utama, set `MAX_OBJECTS_PER_IMAGE = 1` (mengambil deteksi dengan skor tertinggi).
- Jika ada foto yang berisi beberapa pakaian sekaligus (misal kondisi oklusi / menumpuk), ubah `MAX_OBJECTS_PER_IMAGE = 3`.


In [ ]:
MAX_OBJECTS_PER_IMAGE = 2   # Ubah ke 1 jika tiap foto pasti hanya 1 objek utama
BOX_THRESHOLD = 0.25        # Threshold deteksi GroundingDINO
POLY_EPSILON_RATIO = 0.003  # Semakin besar -> semakin sedikit titik poligon (lebih mudah diedit di Roboflow)

# Buat daftar kategori COCO
unique_classes = list(dict.fromkeys(cfg['class_name'] for cfg in FOLDER_CONFIG.values()))
class_to_cat_id = {name: i + 1 for i, name in enumerate(unique_classes)}

coco_output = {
    'info': {'description': 'Pre-annotated HP Test Set (GroundingDINO + SAM)'},
    'licenses': [],
    'categories': [
        {'id': cid, 'name': cname, 'supercategory': 'clothing'}
        for cname, cid in class_to_cat_id.items()
    ],
    'images': [],
    'annotations': [],
}


def mask_to_polygons(bin_mask, epsilon_ratio=0.003, min_area=100):
    contours, _ = cv2.findContours(bin_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue
        eps = epsilon_ratio * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, eps, True)
        if len(approx) >= 3:
            poly = approx.flatten().tolist()
            if len(poly) >= 6:
                polygons.append([float(x) for x in poly])
    return polygons


ann_id = 1
COLORS = [(0, 255, 0), (255, 165, 0), (0, 191, 255)]

for img_id, (img_path, folder_key) in enumerate(tqdm(found_files, desc='Pre-annotating'), start=1):
    cfg = FOLDER_CONFIG[folder_key]
    cls_name = cfg['class_name']
    cat_id = class_to_cat_id[cls_name]
    prompt = cfg['prompt']

    # Salin foto ke folder output (dengan prefix subfolder agar nama file unik)
    out_fname = f'{folder_key}_{os.path.basename(img_path)}'
    out_img_path = os.path.join(OUTPUT_DIR, out_fname)
    shutil.copy2(img_path, out_img_path)

    pil_img = Image.open(img_path).convert('RGB')
    w, h = pil_img.size

    coco_output['images'].append({
        'id': img_id,
        'file_name': out_fname,
        'width': w,
        'height': h,
    })

    # 1. Deteksi box dengan GroundingDINO
    inputs = dino_processor(images=pil_img, text=prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = dino_model(**inputs)

    results = dino_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        box_threshold=BOX_THRESHOLD,
        text_threshold=0.20,
        target_sizes=[(h, w)],
    )[0]

    boxes = results['boxes'].cpu().numpy()
    scores = results['scores'].cpu().numpy()

    # Urutkan berdasarkan skor tertinggi
    if len(scores) > 0:
        order = np.argsort(-scores)[:MAX_OBJECTS_PER_IMAGE]
        boxes = boxes[order]
    else:
        # Fallback: jika foto terlalu gelap/sulit, pakai box tengah 70% gambar untuk SAM
        boxes = np.array([[w * 0.15, h * 0.15, w * 0.85, h * 0.85]], dtype=np.float32)

    # 2. Segmentasi tiap box dengan SAM
    vis_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    overlay = vis_img.copy()

    for b_idx, box in enumerate(boxes):
        input_boxes = [[[float(box[0]), float(box[1]), float(box[2]), float(box[3])]]]
        sam_inputs = sam_processor(pil_img, input_boxes=input_boxes, return_tensors='pt').to(device)
        with torch.no_grad():
            sam_outputs = sam_model(**sam_inputs)

        masks = sam_processor.image_processor.post_process_masks(
            sam_outputs.pred_masks.cpu(),
            sam_inputs['original_sizes'].cpu(),
            sam_inputs['reshaped_input_sizes'].cpu(),
        )[0]  # shape: (1, 3, H, W)

        iou_scores = sam_outputs.iou_scores.cpu().numpy()[0, 0]
        best_mask_idx = int(np.argmax(iou_scores))
        bin_mask = masks[0, best_mask_idx].numpy().astype(np.uint8)

        polygons = mask_to_polygons(bin_mask, epsilon_ratio=POLY_EPSILON_RATIO)
        if not polygons:
            continue

        ys, xs = np.where(bin_mask == 1)
        x1, y1, x2, y2 = float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())
        bbox_coco = [x1, y1, x2 - x1, y2 - y1]
        area = float(bin_mask.sum())

        coco_output['annotations'].append({
            'id': ann_id,
            'image_id': img_id,
            'category_id': cat_id,
            'segmentation': polygons,
            'area': area,
            'bbox': bbox_coco,
            'iscrowd': 0,
        })
        ann_id += 1

        # Gambar preview
        color = COLORS[(cat_id - 1) % len(COLORS)]
        overlay[bin_mask == 1] = color
        cv2.rectangle(vis_img, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        cv2.putText(vis_img, cls_name, (int(x1), max(25, int(y1) - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    blended = cv2.addWeighted(overlay, 0.35, vis_img, 0.65, 0)
    cv2.imwrite(os.path.join(PREVIEW_DIR, out_fname), blended)

coco_json_path = os.path.join(OUTPUT_DIR, '_annotations.coco.json')
with open(coco_json_path, 'w', encoding='utf-8') as f:
    json.dump(coco_output, f, indent=2)

print(f'\nSELESAI!')
print(f'Total gambar    : {len(coco_output["images"])}')
print(f'Total anotasi   : {len(coco_output["annotations"])}')
print(f'Folder Roboflow : {OUTPUT_DIR}')
print(f'Folder Preview  : {PREVIEW_DIR}')


## 4. Cara Upload ke Roboflow & Koreksi Manual
1. Cek dulu hasil preview di Google Drive (`CPMK3_Instance_Segmentation/preannotated_preview/`).
2. Download folder `preannotated_for_roboflow/` (berisi semua foto + `_annotations.coco.json`), atau upload langsung ke proyek **Instance Segmentation** Anda di Roboflow (`Upload Data > Select Folder`).
3. Roboflow akan otomatis membaca semua poligon dari `_annotations.coco.json` beserta kelasnya (`kaos`, `jaket`, `kemeja`).
4. Buka halaman **Annotate** di Roboflow, periksa satu per satu (rapikan poligon yang kurang pas atau hapus objek salah), lalu **ambil screenshot proses anotasi** untuk laporan Anda.
5. Generate Version di Roboflow, lalu lanjut ke notebook `01_setup_and_exif.ipynb`.
